<div dir="rtl">

<h1>פרק 1 · המידע האמיתי</h1>
<p>איך radSeq קורא את RADIal: תוויות, מפה של פריים, ואיך פריימים הופכים לסדרות.</p>
<p><b>כ־15 דקות · 7 שורות אדומות · פרק 1 מתוך 7</b></p>
<p><span style="box-shadow:inset 3px 0 0 rgb(225,70,70);padding:0 6px">אדום</span> יכול לשנות את התוצאה: קוראים כל תו. &nbsp; <span style="box-shadow:inset 3px 0 0 rgb(220,170,30);padding:0 6px">צהוב</span> קוראים רק את הכתובת או הערך. &nbsp; מעומעם: הקשר בלבד. &nbsp; בלוק מקווקו: לא צריך לקרוא.</p>
<p>התאים של "בדיקה" מריצים את הקוד האמיתי על המידע האמיתי. מריצים אותם ומסתכלים על הפלט.</p>

</div>

In [ ]:
import csv, json, os, sys
from pathlib import Path
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "regain" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import radial
from src.viz import show_rd_rows
from regain.view import code, block, RED, YELLOW

<div dir="rtl">

<h2>קודם רואים</h2>
<p>פריים אחד מ־RADIal. העיגולים הם הרכבים המסומנים.</p>

</div>

In [ ]:
by_frame = radial.read_labels()
frames = sorted(by_frame)
s = frames[400]
show_rd_rows({"RADIal": [radial.power_map(s)]},
             marks={"RADIal": [radial.targets(s, by_frame)]},
             col_titles=[f"frame {s}"])

<div dir="rtl">

<h2>1. התוויות</h2>

</div>

In [ ]:
code(radial.read_labels, [
    (YELLOW, 'os.path.join(root, "labels.csv")', "קובץ התוויות, בתיקיית RADIal שעל ה־HPC."),
    (RED, 'if float(r["radar_R_m"]) < 0:', "שורות עם טווח ‎-1 הן תוויות לא תקינות. בלי השורה הזו היו נכנסים רכבים בטווח שלילי."),
])

In [ ]:
# בדיקה: כמה שורות נזרקות
rows = list(csv.DictReader(open(os.path.join(radial.DEFAULT_ROOT, "labels.csv"))))
bad = sum(float(r["radar_R_m"]) < 0 for r in rows)
print(f"{len(rows)} rows, {bad} dropped (radar_R_m < 0), {len(by_frame)} labelled frames")

<div dir="rtl">

<h2>2. מפה של פריים</h2>

</div>

In [ ]:
code(radial.power_map, [
    (YELLOW, 'f"fft_{sample:06d}.npy"', "כל פריים הוא קובץ אחד: מספר הפריים בשש ספרות."),
    (RED, '10 * np.log10((np.abs(x) ** 2).sum(axis=-1)', "סכום ההספק על 16 האנטנות, ואז dB. זה הסולם שכל הפרויקט עובד בו, והסימולטור חייב לעשות בדיוק אותו דבר."),
])

In [ ]:
# בדיקה: מה נכנס ומה יוצא
raw = np.load(os.path.join(radial.DEFAULT_ROOT, "radar_FFT", f"fft_{s:06d}.npy"))
m = radial.power_map(s)
print("on disk:", raw.shape, raw.dtype, "  ->  map:", m.shape, f"{m.min():.1f} .. {m.max():.1f} dB")

<div dir="rtl">

<h2>3. איפה הרכב על המפה</h2>

</div>

In [ ]:
code(radial.targets, [
    (RED, 'float(lab["radar_R_m"]) / DR_M', "0.2 מטר לכל bin. הערך נמדד מהמידע, לא נלקח מהתיעוד."),
    (RED, 'float(lab["radar_D_mps"]) % N_DOPPLER', "למרות השם, העמודה radar_D_mps מכילה מספר bin ולא מהירות. והמיקום הוא של העותק הראשון מתוך 12 (DDMA)."),
])

In [ ]:
# בדיקה: שורת תווית כמו שהיא בקובץ
lab = by_frame[s][0]
print({k: lab[k] for k in ("radar_R_m", "radar_D_mps", "radar_P_db")})

<div dir="rtl">

<h2>4. מפריימים לסדרות</h2>

</div>

In [ ]:
code(radial.sequences, [
    (RED, 'stride = stride or seq_len', "חלונות שלא חופפים: חלונות חופפים מאותה נסיעה הם כמעט כפולים. זו הסיבה שיש מעט סדרות אימון."),
    (RED, 'if cur == prev + 1:', "רצוף פירושו כאן מספרי פריים עוקבים. ההנחה היא שמספרים עוקבים הם 0.2 שניות הפרש. ההנחה הזו לא נבדקה."),
])

In [ ]:
# בדיקה: כמה סדרות אימון יש, בלי חפיפה ועם חפיפה
split = json.loads((ROOT / "data" / "radial" / "split.json").read_text())
recs = radial.recordings()
train = {k: recs[k] for k in split["train"]}
print("8-frame train sequences:", len(radial.sequences(train, seq_len=8)),
      "  with stride 1:", len(radial.sequences(train, seq_len=8, stride=1)))

<div dir="rtl">

<h2>5. החלוקה ל־train / val / test</h2>

</div>

In [ ]:
code(radial.split_recordings, [
    (RED, 'pick = max(remaining, key=lambda k: (remaining[k], k))', "כל הקלטה נכנסת שלמה לחלק אחד, כדי שהבדיקה לא תמדוד שינון. החלוקה נשמרה פעם אחת ב־data/radial/split.json, והקובץ הוא הקובע, לא הפונקציה."),
])

<div dir="rtl">

<h2>בלוקים</h2>
<p>לא צריך לקרוא. אפשר לפתוח אם סקרנים.</p>

</div>

In [ ]:
block(radial.recordings, "מקבץ פריימים לפי שם ההקלטה")
block(radial.load_sequence, "רשימת פריימים ← טנזור (L, 512, 256) ב־dB")
block(radial.range_profile, "חציון ה־dB לכל bin של טווח. נפגוש אותו בפרק 2")
block(radial.background_stats, "הסטטיסטיקה שהסימולטור מותאם אליה. נפגוש אותה בפרק 2")
block(radial.camera_image, "תמונת מצלמה של פריים. לתצוגה בלבד")
block(radial.vehicles, "רכבים בשני המבטים, מצלמה ומפה. לתצוגה בלבד")

<div dir="rtl">

<p><b>סוף פרק 1.</b> הבא: הסימולטור.</p>

</div>